# Evaluate: GSM8K test set

Colab front-end for `src/eval_gsm8k.py`. Greedy decoding, exact-match on the final numeric answer.

**First run the base model**, then each adapter.

## 1. Clone and install

In [1]:
import os

if not os.path.isdir('finetune-gsm8k'):
    !git clone https://github.com/YuZh98/finetune-gsm8k.git
os.chdir('/content/finetune-gsm8k')
!pip install -q -r requirements.txt

Cloning into 'finetune-gsm8k'...
remote: Enumerating objects: 40, done.
remote: Counting objects: 100% (40/40), done.
remote: Compressing objects: 100% (26/26), done.
remote: Total 40 (delta 14), reused 36 (delta 11), pack-reused 0 (from 0)
Receiving objects: 100% (40/40), 21.71 KiB | 2.17 MiB/s, done.
Resolving deltas: 100% (14/14), done.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 751.0/751.0 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 42.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 46.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 52.8 MB/s eta 0:00:00


## 2. Smoke test: 16 problems against the base model

Verify the harness works before running the full 1319 problems.

In [2]:
from src.eval_gsm8k import evaluate

metrics = evaluate(adapter_path=None, limit=16, batch_size=4)
metrics

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

GSM8K eval: 100%|██████████| 4/4 [03:39<00:00, 54.79s/it]


{'n_problems': 16, 'n_correct': 7, 'accuracy': 0.4375, 'elapsed_sec': 219.2}

## 3. Full eval: base model

In [3]:
from src.eval_gsm8k import evaluate, append_row
from src.config import ANSWER_REGEX

metrics = evaluate(adapter_path=None, limit=None, batch_size=8)
append_row('results/runs.csv', {
    'run_id': 'run0_base',
    'adapter': '',
    'answer_regex': ANSWER_REGEX,
    **metrics,
})
metrics

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

GSM8K eval: 100%|██████████| 165/165 [2:20:06<00:00, 50.95s/it]


{'n_problems': 1319,
 'n_correct': 492,
 'accuracy': 0.3730098559514784,
 'elapsed_sec': 8406.8}

## 4. Full eval: a trained adapter

Point `ADAPTER` at the adapter dir saved by the training notebook.

In [4]:
RUN_ID = 'run2_r16'
ADAPTER = f'./runs/{RUN_ID}/adapter'

metrics = evaluate(adapter_path=ADAPTER, limit=None, batch_size=8)
append_row('results/runs.csv', {
    'run_id': RUN_ID,
    'adapter': ADAPTER,
    'answer_regex': ANSWER_REGEX,
    **metrics,
})
metrics

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

ValueError: Can't find 'adapter_config.json' at './runs/run2_r16/adapter'

## 5. Inspect results

In [ ]:
import pandas as pd

df = pd.read_csv('results/runs.csv')
df